# Building a Growable Collection

You will build an array-backed collection that grows while preserving its entries and their order.

CSC-239 · Module 5 · Lesson 3 of 3

You have used arrays and ArrayList, then defined classes with protected state. Here you will connect those ideas by implementing a small collection yourself. The class keeps its storage private and gives callers methods for adding and reading entries. Review the [module vocabulary](terms.md) as needed.


## Learning Goals

- Implement ordered addition using a private backing array and logical size.
- Grow storage without losing entries and test the exact capacity boundary.


## Why This Matters

Applications collect information as people use them: registrations arrive, orders gain items, and reports accumulate results. A fixed array requires someone to decide its length in advance. If callers managed every replacement array themselves, the copying and counting rules would be repeated throughout the application.

A collection class keeps that work behind an operation such as add. Callers describe the change they need, while the class preserves its storage rules. Building this small version connects the ArrayList operations you used earlier with encapsulation. It also prepares you to separate a collection's public contract from its implementation in the next module. This tutorial class demonstrates the mechanism; it does not implement every feature or check of ArrayList.


## Check Your Starting Point

For an array of length two, give its valid indexes and an indexed-loop condition that visits both. Explain why replacing one element does not change the array's length. Recall what assigning another array reference to a variable changes, and why callers use public methods to reach private state. Record your reasoning before opening the answer.


In [ ]:
Your response:

Valid indexes and loop condition:

Element replacement versus array length:

Reference reassignment:

Why use public operations:


<details>
<summary>Show answer</summary>

The indexes are zero and one. Starting index at zero, continuing while index < 2, and adding one each time visits both. Element assignment changes a stored value inside the existing array. Reference assignment changes which array a variable reaches; neither operation changes an existing array’s length. Public operations let the class coordinate changes to private state instead of allowing callers to bypass its rules.

</details>


## Video Demonstration

Follow the constructor, the full-array check and the copy loop. Watch the third name arrive after the original two names have moved into larger storage.

<video controls preload="metadata" width="960">
  <source src="media/03_building_a_growable_collection/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/03_building_a_growable_collection/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the building a growable collection demonstration transcript](media/03_building_a_growable_collection/transcript.md).


## Concept

A campus activity coordinator needs a signup list. Each input is one participant name, and the list must keep names in the order they arrive. The coordinator does not know the final number of participants. Our collection starts with room for two names and makes more room when another name will not fit.

A **backing array** is the array a collection uses internally to store its entries. Its **capacity** is the number of array positions available. The collection's **logical size** is the number of entries actually added. A new collection can therefore have capacity two and size zero. Both positions exist, but neither represents a signup yet.

We keep these facts in separate private fields: `String[] storage` refers to the backing array, and `int size` records the used count. Making the fields private lets the class control changes that must stay consistent. A caller asks `add` to record a name; it does not change storage and size independently.

The used positions run from index zero through `size - 1`. When size is zero, there are no used positions. The next name belongs at index `size`, provided that index is inside the array. This rule connects the count to the insertion position without searching for an empty-looking value.


### Create space before names arrive

The constructor uses `storage = new String[2];` to allocate a two-position array. **Allocation** creates the array with the requested length. The keyword `new` requests the new array, `String` identifies its element type, and the brackets contain its length. Unlike a brace initializer, this expression does not supply participant names.

Java initializes each position of a new reference array to **`null`**, a value that means no object reference is stored there. The expression `storage[0] == null` can test that state. Calling a String method through that position would fail because there is no String object to receive the call. We will read participant names only from positions that our collection has filled.

A new `int` array instead contains zero in every position. Zero can also be a real score someone adds. The stored value alone therefore cannot distinguish an added score from unused capacity. The separate size field tells us which positions belong to the collection.

The constructor also assigns `size = 0;`. Together, these assignments establish our starting rule: two available positions and no added entries. Each new collection receives its own backing array and used count.


In [ ]:
String[] labels = new String[3];
int[] amounts = new int[3];
int used = 0;
System.out.println("Missing: " + (labels[0] == null));
System.out.println("Default amount: " + amounts[0]);
labels[0] = "ink";
amounts[used] = 0;
used = used + 1;
System.out.println("Label: " + labels[0]);
System.out.println("Used: " + used);
System.out.println("Capacity: " + amounts.length);


Expected output:

```text
Missing: true
Default amount: 0
Label: ink
Used: 1
Capacity: 3
```


<details class="animation-panel" open>
<summary>Show or hide animation: array allocation and defaults</summary>
<p><img src="media/03_building_a_growable_collection/array_allocation_and_defaults.gif" alt="New array positions receive defaults; assigning an entry and increasing the used count records a logical addition." width="960" style="max-width:100%;height:auto;"></p>
</details>

New array positions receive defaults; assigning an entry and increasing the used count records a logical addition. This loop lasts 12.4 seconds. Hide it to stop viewing motion, or [view the array allocation and defaults still image](media/03_building_a_growable_collection/array_allocation_and_defaults_still.png).


### Count entries separately from storage

The next complete example uses two positions for a small supply list. Before either entry arrives, size is zero. Store pen at index size, then increase size to one. Store pad at the next index size, then increase size to two. The array length stays two throughout. These additions fit because each insertion index is smaller than the capacity at the moment of assignment.

The reports show size followed by capacity. Start means no entries are used. One means one entry has been added. Full means both positions are used; it does not mean that another array has already been created. Growth is needed only when the next addition arrives.


In [ ]:
String[] storage = new String[2];
int size = 0;
System.out.println("Start: " + size + "/" + storage.length);
storage[size] = "pen";
size = size + 1;
System.out.println("One: " + size + "/" + storage.length);
storage[size] = "pad";
size = size + 1;
System.out.println("Full: " + size + "/" + storage.length);


Expected output:

```text
Start: 0/2
One: 1/2
Full: 2/2
```


<details class="animation-panel" open>
<summary>Show or hide animation: logical size and capacity</summary>
<p><img src="media/03_building_a_growable_collection/logical_size_and_capacity.gif" alt="The used count advances from zero to two while the array keeps its length of two." width="960" style="max-width:100%;height:auto;"></p>
</details>

The used count advances from zero to two while the array keeps its length of two. This loop lasts 12.4 seconds. Hide it to stop viewing motion, or [view the logical size and capacity still image](media/03_building_a_growable_collection/logical_size_and_capacity_still.png).


### Make more room without losing earlier entries

An array's length stays fixed after allocation. Assigning a new name to one position changes that element, not the array's length. When size equals `storage.length`, the next insertion has no available position. We must prepare a larger array before using index size.

Inside the full-array condition, `String[] larger = new String[storage.length * 2];` creates twice as many positions. With capacity two, the new capacity is four. This tutorial starts above zero, so doubling always creates more room. The original array still contains the earlier names; allocating another array does not copy them.

The copy loop starts index at zero and continues while `index < size`. Its assignment, `larger[index] = storage[index];`, reads an existing entry from the old array and stores it at the same index in the new one. Keeping indexes unchanged preserves signup order. For String entries, this copies references to the same String objects; it does not construct new copies of their text.

Only after the loop finishes do we assign `storage = larger;`. That assignment changes which array the field refers to. It does not resize the old array. Later collection operations now use the larger array, which already holds the earlier entries in their original order.

Copying does not add a participant, so size does not change during growth. We then store the incoming name with `storage[size] = name;` and increase size with `size = size + 1;`. The insertion uses the old size as the next free index. Increasing size first would skip that position and could attempt an index outside the array.


In [ ]:
String[] storage = {"oak", "elm"};
int size = 2;
String[] larger = new String[storage.length * 2];
for (int index = 0; index < size; index = index + 1) {
    larger[index] = storage[index];
}
storage = larger;
System.out.println("Capacity: " + storage.length);
for (int index = 0; index < size; index = index + 1) {
    System.out.println(storage[index]);
}


Expected output:

```text
Capacity: 4
oak
elm
```


<details class="animation-panel" open>
<summary>Show or hide animation: copy before replacing storage</summary>
<p><img src="media/03_building_a_growable_collection/copy_before_replacing_storage.gif" alt="Each used reference is copied to the same index before storage is redirected to the larger array." width="960" style="max-width:100%;height:auto;"></p>
</details>

Each used reference is copied to the same index before storage is redirected to the larger array. This loop lasts 14.8 seconds. Hide it to stop viewing motion, or [view the copy before replacing storage still image](media/03_building_a_growable_collection/copy_before_replacing_storage_still.png).


### Grow, append, then increase the count

The next complete example begins with scores six and zero in an array of length two. Both scores are real entries, so size is two. The new score is three. The if condition checks whether size equals the current length before attempting the insertion.

The array is full, so the body allocates four positions, copies the two used values in order, and redirects storage to that array. At this point size is still two. The following assignment stores three at index two, and only then increases size to three. If the array had spare capacity, the program would skip the growth body and perform the same insertion and count update.

The final loop prints each used index beside its score. It stops at size, so it reports three entries, including the added zero at index one. The unused fourth position is outside that loop. This gives stronger evidence than a sum alone, because adding an unused zero would not change a sum.


In [ ]:
int[] storage = {6, 0};
int size = 2;
int value = 3;
if (size == storage.length) {
    int[] larger = new int[storage.length * 2];
    for (int index = 0; index < size; index = index + 1) {
        larger[index] = storage[index];
    }
    storage = larger;
}
storage[size] = value;
size = size + 1;
System.out.println("Size: " + size);
System.out.println("Capacity: " + storage.length);
for (int index = 0; index < size; index = index + 1) {
    System.out.println(index + ": " + storage[index]);
}


Expected output:

```text
Size: 3
Capacity: 4
0: 6
1: 0
2: 3
```


<details class="animation-panel" open>
<summary>Show or hide animation: grow then append then increment</summary>
<p><img src="media/03_building_a_growable_collection/grow_then_append_then_increment.gif" alt="The full score array is copied before inserting the new score at the old size and then increasing the count." width="960" style="max-width:100%;height:auto;"></p>
</details>

The full score array is copied before inserting the new score at the old size and then increasing the count. This loop lasts 17.2 seconds. Hide it to stop viewing motion, or [view the grow then append then increment still image](media/03_building_a_growable_collection/grow_then_append_then_increment_still.png).


## Worked Example

### Keep a signup list in arrival order

The activity coordinator records Maya, Luis, then Nora. Starting capacity is two. The report must show the empty state, the full state after two additions, and the grown state after the third. It must then list all three names in arrival order.


### Give each collection its own storage

The earlier small programs kept storage and size in local variables. The GrowingNames class keeps those same facts in instance fields so they remain available between method calls:

```java
private String[] storage;
private int size;
public GrowingNames() {
    storage = new String[2];
    size = 0;
}
```

The first field refers to this collection's array; the second counts this collection's entries. The private modifier prevents callers from assigning either field directly. Each constructor call allocates a separate two-position array and establishes zero used entries. The constructor has the same name as the class and no return type.

### Put the addition sequence behind one operation

The method header `public void add(String name)` lets a caller supply the next name. Its parameter contains that call's input. Void means there is no returned result: the useful effect is a change to this collection's state.

Inside add, first check `size == storage.length`. Only a full array needs the allocation and ordered copy just demonstrated. Assign the larger reference to storage after copying. Outside that conditional block, assign the incoming name to `storage[size]`, then increase size. Keeping these last two statements outside the condition makes addition work both with and without growth.

### Let the caller inspect added entries

```java
public String get(int index) {
    return storage[index];
}
public int size() {
    return size;
}
public int capacity() {
    return storage.length;
}
```

Get receives an index and returns the String stored at that position. Size returns the field's used count; the parentheses distinguish a method call such as `names.size()` from a field reference inside the class. Capacity returns the array's fixed length. None of these readers adds an entry or changes the array.

The caller then creates one GrowingNames, adds the three names in order, and uses the readers to report what happened. Read the complete class and caller together below.

The `size()` method returns the used count; `capacity()` returns `storage.length`. These methods let us observe both facts without allowing callers to replace the private fields. A report of size three and capacity four means three added names and one spare position.

Our tutorial `get(int index)` returns `storage[index]`. Its caller must supply an index from zero through size minus one. The method does not enforce that collection rule itself. An index can be inside the backing array yet outside the used portion, so using capacity as the traversal bound would read spare positions too.

To list participants, start index at zero and continue while `index < names.size()`. An empty list produces no name lines because the first condition is false. A list with three entries reads indexes zero, one and two, regardless of spare capacity.

Check the empty state, the last addition that fills the array, and the next addition that requires growth. Inspect the ordered entries as well as size and capacity. A backward copy can destroy names while leaving both counts correct. When testing a score collection, an incorrect traversal over spare zeros can even produce the correct total. The number and order of reported entries provide evidence that the total alone cannot supply.


In [ ]:
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
System.out.println("Start: " + names.size() + "/" + names.capacity());
names.add("Maya");
names.add("Luis");
System.out.println("Full: " + names.size() + "/" + names.capacity());
names.add("Nora");
System.out.println("Grown: " + names.size() + "/" + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}


Expected output:

```text
Start: 0/2
Full: 2/2
Grown: 3/4
Maya
Luis
Nora
```

The first two additions fit the original array. Before storing the third name at index 2, add creates a four-position array and copies indexes 0 and 1 into it. The new name follows those copied entries. Logical size becomes 3 while capacity is 4; traversal stops at size and prints only the three added names.


## Guided Practice

Use the worked reasoning to predict, complete, modify, and repair related programs. Record each prediction before its run, then keep your observations separately. Each complete program creates a new collection for that attempt.


### Predict a full collection after growth

Read the complete program without running it. Predict every output line, including the names in order. Track size and capacity after each addition. Mark the addition that needs a larger array and explain why the following addition either grows or fits. Record your prediction before running the next cell or opening the answer.


In [ ]:
Your response:

Predicted output:

Size and capacity after each addition:

Addition that triggers growth and why:

Valid final get indexes:


In [ ]:
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}


### Trace the full and grown list

After running the program described above, record its actual output and explain the state changes in the response below. For tasks with several cases, return to the earlier prediction response before each new run. Keep the original predictions so you can compare them with the results.


In [ ]:
Your response:

Actual output and first difference:

State before growth, after copying, and after insertion:

Why order is preserved:

Why growth leaves size unchanged:

Why insertion precedes increment:

Private fields and public operations:


<details>
<summary>Show answer</summary>

The constructor creates a two-position backing array and sets size to 0. Iris uses index 0 and Owen uses index 1. Before adding Bea, size equals capacity at 2, so add creates a four-position array and copies Iris and Owen to the same indexes. Bea goes at index 2 and size becomes 3. Kai fits at index 3 without another growth. Size and capacity are both 4. The loop reads indexes 0 through 3, which are exactly the four added elements. Every original array keeps its length; assigning storage = larger changes which array the field refers to. The next valid add will grow before writing at index 4. Immediately after copying, size is still 2 and capacity is 4; only Iris and Owen are collection entries. After Bea is stored, size becomes 3. An empty collection has no valid get index. Private storage remains inside the class; the public readers report its size, capacity or a valid logical element.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 4
Capacity: 4
Iris
Owen
Bea
Kai
```

Common error: Treating capacity as the number of names already added. Growing as soon as the array becomes full, instead of before the next addition when full. Assuming the new array already contains the earlier names.

</details>


### Distinguish a default value from an added element

Predict all five lines before running this complete array program. Explain which positions the program explicitly assigns and which keep Java’s default values. Then run it and compare. Why can the two printed integer zeros not tell you which position was explicitly assigned? Connect that limitation to the separate size field in a growable collection. This probe uses raw arrays; its valid array indexes do not make unused positions valid collection elements. Do not call a String method through a null reference.


In [ ]:
Your response:

Predicted five lines:

Explicitly assigned positions versus defaults:


In [ ]:
String[] labels = new String[3];
int[] counts = new int[3];
labels[1] = "ready";
counts[1] = 0;
System.out.println("Slots: " + labels.length);
System.out.println("First missing: " + (labels[0] == null));
System.out.println("Second: " + labels[1]);
System.out.println("First count: " + counts[0]);
System.out.println("Second count: " + counts[1]);


### Distinguish defaults from added values

After running the program described above, record its actual output and explain the state changes in the response below. For tasks with several cases, return to the earlier prediction response before each new run. Keep the original predictions so you can compare them with the results.


In [ ]:
Your response:

Actual five lines and comparison:

Remaining null position:

Why equal zeros do not show assignment history:

How logical size identifies added entries:


<details>
<summary>Show answer</summary>

Each new array has length 3. The String array starts with null in every position; assigning ready at index 1 leaves index 0 null. The int array starts with zero in every position. Explicitly assigning zero at index 1 leaves it numerically equal to the untouched default at index 0. The values alone cannot tell you which assignment occurred. A growable collection therefore tracks its added entries with a separate size instead of searching for null or zero. The boolean comparison checks for null without calling a method through it.

```java
String[] labels = new String[3];
int[] counts = new int[3];
labels[1] = "ready";
counts[1] = 0;
System.out.println("Slots: " + labels.length);
System.out.println("First missing: " + (labels[0] == null));
System.out.println("Second: " + labels[1]);
System.out.println("First count: " + counts[0]);
System.out.println("Second count: " + counts[1]);
```

Expected output:

```text
Slots: 3
First missing: true
Second: ready
First count: 0
Second count: 0
```

Common error: Assuming new String[3] fills positions with empty strings. Assuming a zero value proves no score has been added. Using a default value as a replacement for the logical size.

</details>


### Complete a copy-and-append sequence

The displayed draft is incomplete and for reading only. Copy it into the empty work cell and replace all four placeholders. Choose COPY_TARGET from larger or storage. Choose COPY_SOURCE from storage or larger. Choose NEXT_STORAGE from larger or storage. Choose INSERT_INDEX from size or size + 1. Keep the rest unchanged. The completed program must preserve red and blue in that order and add green after them. Predict the size, capacity and three displayed values, run the complete repair, and explain the copy direction and the order of reference reassignment, insertion and size update.

This sample is for repair:

```java
String[] storage = {"red", "blue"};
int size = 2;
String[] larger = new String[storage.length * 2];
for (int index = 0; index < size; index = index + 1) {
    <COPY_TARGET>[index] = <COPY_SOURCE>[index];
}
storage = <NEXT_STORAGE>;
storage[<INSERT_INDEX>] = "green";
size = size + 1;
System.out.println("Size: " + size);
System.out.println("Capacity: " + storage.length);
for (int index = 0; index < size; index = index + 1) {
    System.out.println(storage[index]);
}
```


In [ ]:
Your response:

Four choices:

Predicted size, capacity and entries:

Copy direction and update order:


### Check the copy and append

After running the program described above, record its actual output and explain the state changes in the response below. For tasks with several cases, return to the earlier prediction response before each new run. Keep the original predictions so you can compare them with the results.


In [ ]:
Your response:

Actual output and comparison:

Where earlier entries moved:

Why reference replacement precedes insertion:

Why insertion uses the old size:

Unused position:


<details>
<summary>Show answer</summary>

COPY_TARGET is larger and COPY_SOURCE is storage: the existing values must move from the old array into the new array. NEXT_STORAGE is larger so later insertion uses the array with four positions. INSERT_INDEX is size, which is 2 before insertion. Green goes after red and blue at index 2. Increasing size afterward gives 3 while capacity stays 4. The final loop visits only the three used positions and leaves the spare default-null position unprinted.

```java
String[] storage = {"red", "blue"};
int size = 2;
String[] larger = new String[storage.length * 2];
for (int index = 0; index < size; index = index + 1) {
    larger[index] = storage[index];
}
storage = larger;
storage[size] = "green";
size = size + 1;
System.out.println("Size: " + size);
System.out.println("Capacity: " + storage.length);
for (int index = 0; index < size; index = index + 1) {
    System.out.println(storage[index]);
}
```

Expected output:

```text
Size: 3
Capacity: 4
red
blue
green
```

Common error: Copying default nulls from the new array over earlier values. Keeping storage attached to the full old array. Using size + 1 for insertion and skipping the next position.

</details>


### Add one entry beyond the new capacity

Run this complete four-name starter and record its report. Insert only `names.add("Zoe");` after the Kai addition and before the two report statements. Keep the class and traversal unchanged. Predict the new complete output before running the whole modified cell. Trace which entries are copied and where Zoe is stored. Explain why the call causes a growth even though the caller does not create an array itself. Remove only the Zoe addition and run the complete restored starter to compare the fourth-addition boundary again.


In [ ]:
Your response:

Predicted starter output:

Predicted fifth-addition output:

Size and capacity before addition:

Copied indexes and Zoe insertion index:


In [ ]:
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}


### Test a second growth boundary

After running the program described above, record its actual output and explain the state changes in the response below. For tasks with several cases, return to the earlier prediction response before each new run. Keep the original predictions so you can compare them with the results.


In [ ]:
Your response:

Actual starter and modified output:

Why add handles growth for its caller:

Why spare positions are omitted:

Restored four-name output:

How each attempt created a new collection:


<details>
<summary>Show answer</summary>

Before the fifth addition, size and capacity are both 4. Adding Zoe triggers allocation of an eight-position array. The loop copies Iris, Owen, Bea and Kai to indexes 0 through 3 in order. Storage then refers to the new array, Zoe is stored at index 4, and size becomes 5. The three remaining positions are spare capacity. No caller-side array work is needed because add contains the growth rule. Restoring the four-addition caller constructs a fresh collection whose final size and capacity are both 4.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
names.add("Zoe");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 5
Capacity: 8
Iris
Owen
Bea
Kai
Zoe
```

Common error: Changing the constructor’s initial capacity instead of exercising growth. Adding Zoe after the report and expecting the earlier report to include her. Reporting all eight capacity positions as collection entries.

**Additional test: `Four additions: Iris, Owen, Bea, Kai`.** The fourth entry fits the four-position array produced by the third addition.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 4
Capacity: 4
Iris
Owen
Bea
Kai
```

</details>


### Repair copying in the wrong direction

The displayed program is intentionally incorrect; do not run it. Its copy assignment is reversed. Trace the third addition and predict the complete faulty report, including the names. Identify which values are overwritten and where the null values come from. Write the complete repaired program in the empty work cell. Change only the copy assignment so each old entry is copied into the same index in larger. Keep the rest of the class, four names and report unchanged. Predict and run the repair. Also test the complete repaired program with only Iris, Owen and Bea, then with no add calls. Keep the report and traversal. Explain why size and capacity alone would miss this copying bug. Restore all four names afterward.

This sample is for repair:

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                storage[index] = larger[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```


In [ ]:
Your response:

Predicted faulty report:

Values overwritten and source of null:

Repaired assignment:

Predicted repaired four-name, three-name and empty reports:


### Check preservation of actual entries

After running the program described above, record its actual output and explain the state changes in the response below. For tasks with several cases, return to the earlier prediction response before each new run. Keep the original predictions so you can compare them with the results.


In [ ]:
Your response:

Actual repaired reports and comparisons:

Why size and capacity alone miss the defect:

Why the empty case makes no get calls:

Restored four-name report:


<details>
<summary>Show answer</summary>

During the third addition, the faulty loop reads null from the newly allocated larger array and writes it over Iris and Owen in storage. It then assigns storage = larger, whose first two positions still hold null. Bea and Kai are added at indexes 2 and 3, so the faulty report has the expected size 4 and capacity 4 but prints null, null, Bea and Kai. Repair the assignment to larger[index] = storage[index]. The loop now preserves Iris and Owen, and all four names appear in order. The three-name case checks the first growth directly. The empty case has size 0, capacity 2 and no valid get index; the traversal performs zero iterations. Encapsulation does not make an incorrect internal copy correct.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
names.add("Kai");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 4
Capacity: 4
Iris
Owen
Bea
Kai
```

Common error: Reversing the reference assignment instead of the element-copy assignment. Treating correct size and capacity as proof that values survived. Testing only two additions, which never execute the copy loop. Calling get(0) on an empty collection to inspect its unused array position.

**Additional test: `Repaired caller with Iris, Owen and Bea only`.** The third addition triggers the first growth and preserves both earlier names.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
names.add("Iris");
names.add("Owen");
names.add("Bea");
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 3
Capacity: 4
Iris
Owen
Bea
```

**Additional test: `Repaired caller with no additions`.** No entry was added. The logical traversal has zero iterations and makes no get call.

```java
class GrowingNames {
    private String[] storage;
    private int size;
    public GrowingNames() {
        storage = new String[2];
        size = 0;
    }
    public void add(String name) {
        if (size == storage.length) {
            String[] larger = new String[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = name;
        size = size + 1;
    }
    public String get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingNames names = new GrowingNames();
System.out.println("Size: " + names.size());
System.out.println("Capacity: " + names.capacity());
for (int index = 0; index < names.size(); index = index + 1) {
    System.out.println(names.get(index));
}
```

Expected output:

```text
Size: 0
Capacity: 2
```

</details>


## Independent Practice

### Build GrowingScores with private backing storage

Build the tutorial class GrowingScores. Use a private int[] storage field and a separate private int size field. Its public no-argument constructor must allocate capacity 2 and initialize size to 0. Write public void add(int score), public int get(int index), public int size(), and public int capacity(). When full, add must allocate an array with twice the current length, copy the used values in order, and replace storage before inserting the new score at index size. Increase size after insertion. The get caller must provide 0 <= index < size; use only valid indexes and do not add exception handling. Create scores, add 4, 0, 7, 2 and 5 in that order, then use a loop over logical indexes to compute total. Print exactly `Size: 5`, `Capacity: 8`, and `Total: 18`. Plan the fields and state rule before writing. Predict the result, run your entire class and caller so each attempt constructs a new collection, and explain why the added zero counts while unused zeros do not. This tutorial class is separate from the graded StudentArrayList assignment.


In [ ]:
Your response:

Private fields and state rule:

Constructor initialization:

Growth, copy and insertion plan:

Predicted report:


### Explain the score collection

After running the program described above, record its actual output and explain the state changes in the response below. For tasks with several cases, return to the earlier prediction response before each new run. Keep the original predictions so you can compare them with the results.


In [ ]:
Your response:

Actual report and comparison:

Why an added zero counts:

Why spare zeros do not count:

How ordered entries were preserved:

Valid get indexes:


### Test empty, full and grown collections

Test each required case: no additions; 4 and 0; 4, 0 and 7; 4, 0, 7 and 2; 4, 0, 7, 2 and 5; and two zeros with your complete GrowingScores class and caller. Keep the class, logical traversal and three report statements unchanged; change only the add calls. Run the whole cell for each case so its constructor creates a new collection, rather than adding onto a previous test. Predict all three report lines before each run. Afterward record the actual output and explain each size/capacity boundary. Compare the empty case with the two-zero case: both totals can be zero while logical sizes differ. Then restore the exact five additions. After its three report lines, add a second loop over indexes below scores.size() that prints `Score `, the index, `: ` and scores.get(index). Predict and compare all five indexed values in insertion order. Count exactly five diagnostic lines, ending at index 4, and check that no indexes 5 through 7 are reported. Adding unused zero-filled positions could leave Total: 18 unchanged, so that total alone does not prove the traversal bound is correct. Explain why a correct sum alone cannot prove the order or show whether a zero was counted. Repair any mismatch and repeat the cases. Remove the diagnostic loop when finished and restore the exact contracted five-addition output. Never call get for a negative index, an index equal to size, or any index on an empty collection.


In [ ]:
Your response:

Predicted Size / Capacity / Total for each case:
None:
4, 0:
4, 0, 7:
4, 0, 7, 2:
4, 0, 7, 2, 5:
0, 0:

Predicted five indexed diagnostic lines:


### Compare all boundary cases

After running the program described above, record its actual output and explain the state changes in the response below. For tasks with several cases, return to the earlier prediction response before each new run. Keep the original predictions so you can compare them with the results.


In [ ]:
Your response:

Actual report and match or repair for each case:
None:
4, 0:
4, 0, 7:
4, 0, 7, 2:
4, 0, 7, 2, 5:
0, 0:

Actual diagnostic lines, line count and final index:

Why unused zeros could leave total unchanged:

Why additions three and five grow but two and four fit:

Empty versus two-zero size and total:

Why totals cannot prove order:

How each test starts fresh:

Corrections and repeated cases:

Restored report after removing diagnostics:


<details>
<summary>Show answer</summary>

GrowingScores keeps the backing int array and logical size private. Its constructor allocates two positions and explicitly sets size to 0. The five additions put 4, 0, 7, 2 and 5 at logical indexes 0 through 4. The third addition copies the first two scores into capacity 4; the fifth copies the first four into capacity 8. Each new score is stored at the old size, then size increases by one. The caller reads only indexes below scores.size(), so total is 4 + 0 + 7 + 2 + 5 = 18. The added zero at index 1 counts as an element. The unused zero-filled positions at indexes 5 through 7 do not count, because they lie at or beyond logical size. A total alone cannot reveal whether the zero was counted or whether entries were reordered; check size and the ordered values too. The get method assumes 0 <= index < size and does not enforce that rule itself. Summing the three unused zero-filled array positions could also leave 18 unchanged. The valid diagnostic prints exactly five entries at indexes 0 through 4, so its line count and final index provide evidence about the logical traversal bound. The boundary cases check a new collection, a full initial array, the first growth, a full four-position array and the second growth. The two-zero case separates the count of additions from the total. The indexed report checks the values and insertion order after both copies; size, capacity and total alone are not enough to establish all three.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
scores.add(2);
scores.add(5);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 5
Capacity: 8
Total: 18
```

Common error: Using capacity as the traversal limit. Skipping an added score because its value is zero. Returning storage.length from size(). Appending before making space in a full array. Assuming the tutorial operations define the graded assignment interface.

**Additional test: No additions.** The constructor reserves two positions but adds no elements. The loop has zero iterations and makes no get call.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 0
Capacity: 2
Total: 0
```

**Additional test: Add 4, then 0.** Both additions fit. The explicit zero is still an added element, so size is 2.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 2
Capacity: 2
Total: 4
```

**Additional test: Add 4, 0, 7.** The third addition grows before writing at index 2 and preserves 4 and 0.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 3
Capacity: 4
Total: 11
```

**Additional test: Add 4, 0, 7, 2.** The fourth addition fills the four-position array; it does not yet require another allocation.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
scores.add(2);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 4
Capacity: 4
Total: 13
```

**Additional test: Add 0, then 0.** Two zero additions have the same total as no additions, but their logical size is 2 rather than 0.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(0);
scores.add(0);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
```

Expected output:

```text
Size: 2
Capacity: 2
Total: 0
```

**Additional test: Exact contracted program followed by the indexed diagnostic loop.** The five valid get calls establish insertion order and the explicitly added zero after both growth events. The extra diagnostic is removed to restore the contracted three-line report. Exactly five indexed lines end at index 4; indexes 5 through 7 must not appear. Adding their unused zeros would not change the total, so the sum is insufficient evidence.

```java
class GrowingScores {
    private int[] storage;
    private int size;
    public GrowingScores() {
        storage = new int[2];
        size = 0;
    }
    public void add(int score) {
        if (size == storage.length) {
            int[] larger = new int[storage.length * 2];
            for (int index = 0; index < size; index = index + 1) {
                larger[index] = storage[index];
            }
            storage = larger;
        }
        storage[size] = score;
        size = size + 1;
    }
    public int get(int index) {
        return storage[index];
    }
    public int size() {
        return size;
    }
    public int capacity() {
        return storage.length;
    }
}
GrowingScores scores = new GrowingScores();
scores.add(4);
scores.add(0);
scores.add(7);
scores.add(2);
scores.add(5);
int total = 0;
for (int index = 0; index < scores.size(); index = index + 1) {
    total = total + scores.get(index);
}
System.out.println("Size: " + scores.size());
System.out.println("Capacity: " + scores.capacity());
System.out.println("Total: " + total);
for (int index = 0; index < scores.size(); index = index + 1) {
    System.out.println("Score " + index + ": " + scores.get(index));
}
```

Expected output:

```text
Size: 5
Capacity: 8
Total: 18
Score 0: 4
Score 1: 0
Score 2: 7
Score 3: 2
Score 4: 5
```

</details>


## Summary

Logical size counts used elements; capacity counts available array positions. New arrays receive default element values, so unused capacity needs a separate used count. A growing collection allocates larger storage, copies used entries in order, and then appends the new entry. Each individual array keeps its fixed length.


### Recall the growth sequence

Without looking back, explain size versus capacity and trace the order of allocation, copying, reference replacement, insertion and count update. Give one common error and a check that reveals it.


In [ ]:
Your response:

Size versus capacity:

Growth sequence and reason for its order:

Common error and revealing check:


<details>
<summary>Show answer</summary>

Size counts added entries; capacity counts array positions. Allocate larger storage and copy used entries before replacing the storage reference. Insert at the old size, then increment size. Reversing the copy direction loses entries: check their actual values and order, not just the counts.

</details>


## Reflection

Explain how a growing signup list can preserve the order students joined. Describe tests for an empty list, a full backing array and one new entry beyond that boundary. State what a caller needs to know about valid indexes.


In [ ]:
Your response:

How signup order is preserved:

Empty, full and beyond-full tests:

Caller responsibility for get indexes:


The next module separates the operations a class promises from the code that implements them, using interfaces and shared behavior.


## Supplemental Reading

- [Java 21 arrays](https://docs.oracle.com/javase/specs/jls/se21/html/jls-10.html) defines array creation, fixed length and element access.
- [Java 21 initial values](https://docs.oracle.com/javase/specs/jls/se21/html/jls-4.html#jls-4.12.5) specifies default array-element values and distinguishes them from local-variable initialization.
